# Naive RAG — chunk · embed · top-k · stuff

The simplest retrieval-augmented pipeline:

1. **Chunk** the corpus (here: one chunk per abstract — they're already short).
2. **Embed** each chunk with a bi-encoder. We use a deterministic hashing embedder so this notebook runs offline; swap in `shared.llm.embed(...)` for a real provider.
3. **Top-k retrieve** by cosine similarity over the question embedding.
4. **Stuff** the top-k chunks into a single LLM prompt and read the answer.

This is the baseline against which every other RAG technique in `01-rag/` is measured (`12-comparison-bench/` aggregates all snapshots).

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa
from shared.prompts import RAG_SYSTEM, rag_user_prompt

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/00-naive-rag'
K = 3
EMBED_DIMS = 256
EMBED_SEED = 0

corpus = load_corpus()
qa = load_golden_qa()
print('corpus:', len(corpus), 'docs   |   golden Q&A:', len(qa))

corpus: 10 docs   |   golden Q&A: 30


## Index the corpus
One vector per document, L2-normalized. Indexing 10 abstracts is instantaneous; indexing a real ~500-paper corpus takes a few seconds.

In [3]:
import numpy as np
doc_texts = [d.title + '. ' + d.abstract for d in corpus]
doc_vecs = hash_embed(doc_texts, dims=EMBED_DIMS, seed=EMBED_SEED)
print('index shape:', doc_vecs.shape, '  norms:', np.linalg.norm(doc_vecs, axis=1)[:3].round(4))

index shape: (10, 256)   norms: [1. 1. 1.]


## Retrieve top-k for a question
We embed the question with the *same* embedder, then take the k highest cosine scores.

In [4]:
def retrieve(question: str, k: int = K) -> list[tuple[str, str]]:
    q_vec = hash_embed([question], dims=EMBED_DIMS, seed=EMBED_SEED)[0]
    idx, scores = cosine_topk(q_vec, doc_vecs, k=k)
    return [(corpus[i].arxiv_id, corpus[i].title + '. ' + corpus[i].abstract) for i in idx]

for doc_id, _ in retrieve('How much does RA-MoE reduce latency?'):
    print(' ', doc_id)

  synth-007
  synth-004
  synth-006


## Generate an answer
Stuff the top-k chunks into the prompt and let the model draft an answer.

In [5]:
def answer(question: str) -> tuple[str, list[str]]:
    contexts = retrieve(question)
    reply = complete(
        model=MODEL, namespace=NS,
        messages=[
            Message(role='system', content=RAG_SYSTEM),
            Message(role='user', content=rag_user_prompt(question, contexts)),
        ],
    )
    return reply.content, [doc_id for doc_id, _ in contexts]

demo_q = qa[0].question
ans, ctx = answer(demo_q)
print('Q:', demo_q)
print('ANSWER:', ans)
print('USED:', ctx)

Q: By how much does RA-MoE reduce p99 decode latency compared to standard learned routing?
ANSWER: 38%.
USED: ['synth-001', 'synth-008', 'synth-004']


## Sweep the golden set
We loop through every question and collect the answer for inspection.

In [6]:
rows = []
for item in qa:
    ans, ctx = answer(item.question)
    rows.append({
        'id': item.id,
        'question': item.question,
        'tag': item.tags[0] if item.tags else '',
        'sources': item.source_ids,
        'retrieved': ctx,
        'answer': ans,
    })
print('answered', len(rows), 'questions')

answered 30 questions


In [7]:
# Quick visual: first 3 results.
for r in rows[:3]:
    print(f"\n[{r['id']}] tag={r['tag']}")
    print('Q:', r['question'])
    print('retrieved:', r['retrieved'])
    print('answer:', r['answer'])


[q01] tag=direct
Q: By how much does RA-MoE reduce p99 decode latency compared to standard learned routing?
retrieved: ['synth-001', 'synth-008', 'synth-004']
answer: 38%.

[q02] tag=direct
Q: How many experts does the RA-MoE 47B model use, and what is the routing strategy?
retrieved: ['synth-001', 'synth-003', 'synth-010']
answer: 32 experts with top-2 routing.

[q03] tag=direct
Q: What does the RA-MoE paper say about the routing entropy regularizer used in prior MoE work?
retrieved: ['synth-001', 'synth-008', 'synth-007']
answer: It harms latency; the paper proposes a stability-only auxiliary loss instead.


## Eval
See `eval.py` in this folder for the full metrics computation. It writes `eval-snapshot.json`, which is committed and tracked by CI for regressions.

Metrics tracked:
- **context_recall** — fraction of source documents that appear in the retrieved set.
- **refusal_rate_on_unanswerable** — fraction of unanswerable questions where the model correctly refused.
- **answer_exact_match_direct** — fraction of direct questions whose answer is the reference (substring).
- **n_queries** — denominator for everything above.

## When NOT to use naive RAG
* When chunks are long and dense — semantic chunking helps (see `01-chunking-strategies/`).
* When recall@k is the bottleneck — try hybrid search (`03-hybrid-search/`).
* When the top-k contains near-duplicates — add reranking (`04-reranking/`).
* When the question requires multi-hop reasoning — try `08-agentic-rag/` or `09-graph-rag/`.